# 03 — Groq RAG Pipeline
**Phase:** Foundation — Day 2

**What this notebook does:**
- Connects ChromaDB retrieval to LLaMA-3 via Groq (free)
- Builds a strict prompt that enforces citations
- Implements basic hallucination guard
- Tests 3 query types: answerable, specific, unanswerable

**Requires:** `.env` file with `GROQ_API_KEY=your_key`

In [ ]:
!pip install langchain-groq langchain python-dotenv -q

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

print(f"API key loaded: {GROQ_API_KEY[:8]}...")
print("All imports successful")

## Step 1 — Reconnect to ChromaDB

In [ ]:
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Chunks available: {collection.count()}")

## Step 2 — Initialise the LLM
`temperature=0.0` means deterministic — no creativity, just facts from context.

In [ ]:
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.0,
    api_key=GROQ_API_KEY
)

# Quick connectivity test
test = llm.invoke([HumanMessage(content="Reply with exactly: LLM connected")])
print(test.content)

## Step 3 — Retrieval function

In [ ]:
def retrieve_chunks(query, top_k=3):
    query_vec = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k
    )
    chunks    = results["documents"][0]
    ids       = results["ids"][0]
    distances = results["distances"][0]
    return chunks, ids, distances

## Step 4 — Prompt builder
This prompt is your hallucination guard. It tells the LLM:
1. Only use the provided context
2. Decline if unsure
3. Always cite the source chunk

In [ ]:
SYSTEM_PROMPT = """You are a precise document assistant for DocMind.
Answer the user's question using ONLY the context chunks provided below.

Rules you must follow:
1. Use ONLY information from the provided context. Never use outside knowledge.
2. If the context does not contain enough information to answer, respond with:
   "I don't have enough information in the provided documents to answer this."
3. Always end your answer with: Source: [chunk_id]
4. Be concise and direct. No filler phrases.
5. If multiple chunks support the answer, cite all of them: Source: [chunk_0, chunk_3]"""


def build_prompt(query, chunks, ids):
    # Label each chunk with its ID so LLM knows what to cite
    context = ""
    for chunk, cid in zip(chunks, ids):
        context += f"\n[{cid}]\n{chunk}\n"

    human_content = f"""Context:
{context}
---
Question: {query}

Answer (must cite chunk ID at the end):"""

    return [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=human_content)
    ]


# Preview the prompt — always check what you send to the LLM
sample_chunks, sample_ids, _ = retrieve_chunks("What is this document about?")
messages = build_prompt("What is this document about?", sample_chunks, sample_ids)
print("--- SYSTEM MESSAGE ---")
print(messages[0].content)
print("\n--- HUMAN MESSAGE (first 600 chars) ---")
print(messages[1].content[:600])

## Step 5 — Full RAG pipeline function

In [ ]:
# Distance threshold — if all chunks score above this, the context is too weak
DISTANCE_THRESHOLD = 0.8

def rag_answer(query, verbose=True):
    print(f"\n{'='*60}")
    print(f" Question: {query}")
    print(f"{'='*60}")

    # 1. Retrieve
    chunks, ids, distances = retrieve_chunks(query, top_k=3)

    if verbose:
        print("\nRetrieved chunks:")
        for cid, dist in zip(ids, distances):
            quality = 'strong' if dist < 0.4 else 'ok' if dist < 0.8 else 'weak'
            print(f"  {cid}  distance={dist:.4f}  [{quality}]")

    # 2. Hallucination guard — weak context = decline before calling LLM
    if min(distances) > DISTANCE_THRESHOLD:
        answer = "I don't have enough information in the provided documents to answer this."
        print(f"\n[Guard triggered — best distance {min(distances):.4f} > threshold {DISTANCE_THRESHOLD}]")
        print(f"\nAnswer:\n{answer}")
        return answer

    # 3. Build prompt
    messages = build_prompt(query, chunks, ids)

    # 4. Call LLM
    response = llm.invoke(messages)
    answer = response.content

    # 5. Show result
    print(f"\nAnswer:\n{answer}")

    if verbose:
        usage = response.response_metadata.get('token_usage', {})
        print(f"\nTokens — prompt: {usage.get('prompt_tokens','?')} | "
              f"completion: {usage.get('completion_tokens','?')} | "
              f"total: {usage.get('total_tokens','?')}")

    return answer

## Step 6 — Run 3 test queries
**Important:** Change queries 1 and 2 to match your actual PDF content.

In [ ]:
# Test 1: General question the document CAN answer
rag_answer("What is the main topic of this document?")

In [ ]:
# Test 2: Specific question from your PDF
# CHANGE THIS to a specific term or concept from your document
rag_answer("Describe the methodology used in this paper")

In [ ]:
# Test 3: Question the document CANNOT answer
# This should trigger the hallucination guard
rag_answer("What is the population of Mars?")

## Key observations
- Test 3 should always return the refusal message — screenshot this for LinkedIn
- If Test 1/2 answers are wrong, check the retrieved chunk distances
- If distances are all > 0.6, your PDF may have extraction issues
- Token count shows how much of Groq's free quota each call uses

**Next:** `04_hybrid_retrieval.ipynb` — combine BM25 + vector search for better precision.